In [76]:
import sqlite3
import pandas as pd

In [ ]:
conn = sqlite3.connect("chinook.db")
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)

Album = pd.read_sql_query("SELECT * FROM Album", conn)
Artist = pd.read_sql_query("SELECT * FROM Artist", conn)
Customer = pd.read_sql_query("SELECT * FROM Customer", conn)
Employee = pd.read_sql_query("SELECT * FROM Employee", conn)
Genre = pd.read_sql_query("SELECT * FROM Genre", conn)
Invoice = pd.read_sql_query("SELECT * FROM Invoice", conn)
InvoiceLine = pd.read_sql_query("SELECT * FROM InvoiceLine", conn)
MediaType = pd.read_sql_query("SELECT * FROM MediaType", conn)
Playlist = pd.read_sql_query("SELECT * FROM Playlist", conn)
PlaylistTrack = pd.read_sql_query("SELECT * FROM PlaylistTrack", conn)
Track = pd.read_sql_query("SELECT * FROM Track", conn) 

In [83]:
sql_query = ''' select 
                    t.Name AS track_Name,
                    ar.Name AS artist_Name,
                    g.Name AS genre_Name,
                    SUM(i.Quantity) AS total_quantity
                FROM Track as t INNER JOIN InvoiceLine AS i
                ON t.TrackId = i.TrackId INNER JOIN Album AS al 
                ON al.AlbumId = t.AlbumId INNER JOIN Artist AS ar
                ON ar.ArtistId = al.ArtistId INNER JOIN Genre AS g
                ON g.GenreId = t.GenreId
                GROUP BY t.TrackId, t.Name, ar.Name, g.Name
                Having total_quantity > 0
                ORDER BY total_quantity DESC;
            '''

result = pd.read_sql(sql_query,conn)
result

,track_Name,artist_Name,genre_Name,total_quantity
0,Balls to the Wall,Accept,Rock,2
1,Inject The Venom,AC/DC,Rock,2
2,Snowballed,AC/DC,Rock,2
3,Overdose,AC/DC,Rock,2
4,Deuces Are Wild,Aerosmith,Rock,2
...,...,...,...,...
1979,Sing Joyfully,The King's Singers,Classical,1
1980,"Metopes, Op. 29: Calypso",Martin Roscoe,Classical,1
1981,"Symphony No. 2, Op. 16 - ""The Four Temperamen...",Göteborgs Symfoniker & Neeme Järvi,Classical,1
1982,"Étude 1, In C Major - Preludio (Presto) - Liszt",Michele Campanella,Classical,1


In [90]:
Df = (
    Track
    .merge(
        InvoiceLine,
        on='TrackId',
        how='inner',
        suffixes=('_Track', '_InvoiceLine')
    )
    .merge(
        Album,
        on='AlbumId',
        how='inner',
        suffixes=('', '_Album')
    )
    .merge(
        Artist,
        on='ArtistId',
        how='inner',
        suffixes=('', '_Artist')
    )
    .merge(
        Genre,
        on='GenreId',
        how='inner',
        suffixes=('', '_Genre')
    )
)
Df = Df.rename(columns= {"Name" : "Name_Track"})
Result = Df.groupby(['TrackId','Name_Track','Name_Artist','Name_Genre'], as_index = False)['Quantity'].sum().rename(columns = {'Name_Track': 'Track_Name',
           'Name_Artist': 'Artist_Name',
           'Name_Genre': 'Genre_Name',
           'Quantity': 'Total_quantity'})

Result = Result[Result['Total_quantity'] > 0]
Result = Result.sort_values(by = "Total_quantity", ascending= False).reset_index()
Result = Result[['Track_Name',	'Artist_Name',	'Genre_Name',	'Total_quantity']]
Result


,Track_Name,Artist_Name,Genre_Name,Total_quantity
0,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",Emerson String Quartet,Classical,2
1,Somebody To Love,Queen,Rock,2
2,Peace On Earth,U2,Rock,2
3,When I Look At The World,U2,Rock,2
4,Gangland,Iron Maiden,Metal,2
...,...,...,...,...
1979,Transylvania,Iron Maiden,Metal,1
1980,Prowler,Iron Maiden,Metal,1
1981,The Trooper,Iron Maiden,Metal,1
1982,Lord of Light,Iron Maiden,Rock,1
